# 07 — Machine Learning: Airport Clustering (Unsupervised)

**Airline Operations Intelligence Platform** · Notebook 7 of 10 · *runs locally*

## Purpose
Module 10 of the plan: group airports into **operational profiles** using K-Means
(Spark MLlib). Unlike notebook 06 there is no label — the algorithm finds structure on its own.

## Why this is useful, not decorative
"ATL is a high-traffic hub with moderate delays" is a more useful statement to an operations
analyst than "ATL ranks 47th". Clustering turns 322 individual airports into a handful of
interpretable **types**, which the dashboard uses to colour the map and explain each airport
in context.

## Two decisions that determine whether this works

**1. Scaling is mandatory.** `total_flights` ranges from ~50 to ~300,000 while
`delay_rate` is 0–100. K-Means minimises Euclidean distance, so without standardisation
the volume feature dominates completely and the algorithm just sorts airports by size.

**2. Small airports are excluded from clustering, not from the data.** An airport with 200
flights has a delay rate driven by noise. Clustering it produces a confident-looking but
meaningless assignment. Those airports keep every metric and receive `cluster_id = null` —
the same small-sample bias control applied to rankings in notebooks 04 and 05.

In [ ]:
import sys, time, json
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = build_spark("07-clustering")

airports = spark.read.parquet(str(PATHS["marts"] / "airport_metrics.parquet"))
print(f"Airports in mart : {airports.count()}")
airports.printSchema()

---
## 1. Feature selection

Six operational characteristics, all already computed in notebook 05. They describe *how an
airport behaves*, not where it is or how it is named.

In [ ]:
FEATURES = ["avg_dep_delay", "delay_rate", "total_flights",
            "cancellation_rate", "peak_hour_congestion_ratio", "airlines_served"]

eligible = airports.filter(F.col("meets_min_sample"))
print(f"Eligible for clustering (>= 10,000 flights) : {eligible.count()}")
print(f"Excluded (retained, cluster_id = null)      : {airports.count() - eligible.count()}")

# Any nulls would silently drop rows in VectorAssembler.
nulls = eligible.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in FEATURES])
nulls.show()

clustering_input = eligible.na.drop(subset=FEATURES).cache()
print(f"Rows entering K-Means : {clustering_input.count()}")

In [ ]:
clustering_input.select(FEATURES).describe().show()
print("Note the scale difference: total_flights is in the hundreds of thousands while")
print("the rate columns are 0-100. This is why standardisation is not optional.")

---
## 2. Assemble and standardise

`StandardScaler` centres each feature to mean 0 and scales to unit variance, so every
feature contributes equally to the distance metric.

In [ ]:
assembler = VectorAssembler(inputCols=FEATURES, outputCol="raw_features")
assembled = assembler.transform(clustering_input)

scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                        withMean=True, withStd=True)
scaler_model = scaler.fit(assembled)
scaled = scaler_model.transform(assembled).cache()

print("Feature means before scaling :",
      [round(float(v), 2) for v in scaler_model.mean])
print("Feature stddevs before scaling:",
      [round(float(v), 2) for v in scaler_model.std])
scaled.select("features").show(3, truncate=False)

---
## 3. Choosing k — elbow method and silhouette

Two independent criteria, as the plan requires:

- **WCSS** (within-cluster sum of squares) always falls as k rises; the "elbow" is where
  the improvement flattens.
- **Silhouette** measures how well-separated the clusters are, from -1 to 1. Unlike WCSS
  it has an interior optimum, so it can actually select a k.

In [ ]:
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette",
                                distanceMeasure="squaredEuclidean")

search = []
for k in range(2, 11):
    km = KMeans(featuresCol="features", k=k, seed=42, maxIter=50)
    model = km.fit(scaled)
    preds = model.transform(scaled)
    search.append({
        "k": k,
        "wcss": float(model.summary.trainingCost),
        "silhouette": float(evaluator.evaluate(preds)),
        "sizes": sorted(model.summary.clusterSizes, reverse=True),
    })

print(f"{'k':>3}{'WCSS':>12}{'SILHOUETTE':>13}   CLUSTER SIZES")
print("-" * 62)
for r in search:
    print(f"{r['k']:>3}{r['wcss']:>12.1f}{r['silhouette']:>13.4f}   {r['sizes']}")

In [ ]:
# Text elbow plot -- WCSS drop per additional cluster.
print("Elbow: relative WCSS reduction from k-1 to k\n")
print(f"{'k':>3}{'WCSS':>12}{'DROP':>10}{'DROP %':>9}")
print("-" * 36)
prev = None
for r in search:
    if prev is None:
        print(f"{r['k']:>3}{r['wcss']:>12.1f}{'':>10}{'':>9}")
    else:
        drop = prev - r["wcss"]
        print(f"{r['k']:>3}{r['wcss']:>12.1f}{drop:>10.1f}{100*drop/prev:>8.1f}%")
    prev = r["wcss"]

In [ ]:
best = max(search, key=lambda r: r["silhouette"])
print(f"Best silhouette : k = {best['k']}  (score {best['silhouette']:.4f})")
print()
print("Silhouette often favours very small k because it rewards tight, well-separated")
print("blobs. For a dashboard we also need clusters to be INTERPRETABLE and to actually")
print("distinguish airport types, so k is chosen with both criteria plus centroid")
print("inspection below -- not by silhouette alone.")

K = 4     # revisit after inspecting the centroids in section 5
print(f"\nProceeding with k = {K}")

---
## 4. Fit the final model

In [ ]:
t0 = time.time()
kmeans = KMeans(featuresCol="features", predictionCol="cluster_id",
                k=K, seed=42, maxIter=100)
model = kmeans.fit(scaled)
t_fit = time.time() - t0

clustered = model.transform(scaled).cache()
silhouette = evaluator.evaluate(clustered.withColumnRenamed("cluster_id", "prediction"))

print(f"Training time  : {t_fit:.1f}s")
print(f"WCSS           : {model.summary.trainingCost:.1f}")
print(f"Silhouette     : {silhouette:.4f}")
print(f"Cluster sizes  : {model.summary.clusterSizes}")

---
## 5. Interpreting the clusters

A cluster id is meaningless in a dashboard. Centroids are inspected in **original units**
(by inverting the scaling) so each cluster can be given a human label.

In [ ]:
profile = (clustered.groupBy("cluster_id")
    .agg(F.count("*").alias("airports"),
         F.round(F.avg("total_flights"), 0).alias("avg_flights"),
         F.round(F.avg("avg_dep_delay"), 2).alias("avg_dep_delay"),
         F.round(F.avg("delay_rate"), 2).alias("delay_rate"),
         F.round(F.avg("cancellation_rate"), 2).alias("cancellation_rate"),
         F.round(F.avg("peak_hour_congestion_ratio"), 4).alias("peak_congestion"),
         F.round(F.avg("airlines_served"), 1).alias("airlines"))
    .orderBy("cluster_id"))

profile.show(truncate=False)

In [ ]:
# Which airports are in each cluster? Largest by volume, for recognisability.
for row in profile.collect():
    cid = row["cluster_id"]
    members = (clustered.filter(F.col("cluster_id") == cid)
               .orderBy(F.desc("total_flights"))
               .select("airport_code", "total_flights", "delay_rate")
               .limit(8).collect())
    names = ", ".join(f"{m['airport_code']}" for m in members)
    print(f"Cluster {cid}  ({row['airports']:>3} airports, "
          f"avg {int(row['avg_flights']):,} flights, {row['delay_rate']}% delayed)")
    print(f"   largest: {names}\n")

In [ ]:
# Assign labels from the centroid profile: volume tier x delay tier.
stats = profile.collect()
med_vol   = sorted(r["avg_flights"] for r in stats)[len(stats)//2]
med_delay = sorted(r["delay_rate"]  for r in stats)[len(stats)//2]

def label_for(r):
    big  = r["avg_flights"] >= med_vol
    late = r["delay_rate"]  >= med_delay
    if big and late:      return "High-traffic hub, elevated delays"
    if big and not late:  return "High-traffic hub, well-managed"
    if not big and late:  return "Smaller airport, elevated delays"
    return "Smaller airport, reliable"

labels = {r["cluster_id"]: label_for(r) for r in stats}
for cid, lab in sorted(labels.items()):
    print(f"  cluster {cid} -> {lab}")

label_expr = F.create_map([F.lit(x) for kv in labels.items() for x in kv])

---
## 6. Write results back to the mart

`airport_metrics` gains `cluster_id` and `cluster_label`, matching proposal §17. Airports
below the sample threshold keep every metric and receive nulls for the cluster fields.

In [ ]:
assignments = clustered.select("airport_code", "cluster_id") \
                       .withColumn("cluster_label", label_expr[F.col("cluster_id")])

enriched = airports.join(assignments, "airport_code", "left")

print(f"Airports total          : {enriched.count()}")
print(f"With a cluster          : {enriched.filter(F.col('cluster_id').isNotNull()).count()}")
print(f"Without (below sample)  : {enriched.filter(F.col('cluster_id').isNull()).count()}")

# Every eligible airport must have been assigned.
missing = enriched.filter(F.col("meets_min_sample") & F.col("cluster_id").isNull()).count()
assert missing == 0, f"{missing} eligible airports were not clustered"
print("\nAssertion passed: every eligible airport received a cluster.")

In [ ]:
import shutil

# Spark refuses to overwrite a path it is also reading from -- the enriched DataFrame
# still has airport_metrics.parquet in its lineage. Write to a temporary directory,
# then swap. This is also safer: a failure mid-write cannot corrupt the existing mart.
final_path = PATHS["marts"] / "airport_metrics.parquet"
tmp_path   = PATHS["marts"] / "_airport_metrics_tmp.parquet"

enriched.coalesce(1).write.mode("overwrite").parquet(str(tmp_path))

shutil.rmtree(final_path)
shutil.move(str(tmp_path), str(final_path))
print("airport_metrics.parquet updated with cluster_id + cluster_label")

results = (profile.withColumn("cluster_label", label_expr[F.col("cluster_id")])
                  .withColumn("k", F.lit(K))
                  .withColumn("silhouette", F.lit(round(silhouette, 4)))
                  .withColumn("wcss", F.lit(round(float(model.summary.trainingCost), 2))))

results.coalesce(1).write.mode("overwrite") \
       .parquet(str(PATHS["marts"] / "ml_clustering_results.parquet"))
print("ml_clustering_results.parquet written")

(PATHS["marts"] / "ml_clustering_search.json").write_text(json.dumps(search, indent=2))
print("ml_clustering_search.json written (elbow/silhouette search, for the report)")

In [ ]:
model.write().overwrite().save(str(PATHS["models"] / "kmeans_airport_clusters"))
scaler_model.write().overwrite().save(str(PATHS["models"] / "airport_scaler"))
print("Models saved.")

---
## 7. Validation and honest limitations

In [ ]:
check = spark.read.parquet(str(PATHS["marts"] / "airport_metrics.parquet"))
print(f"Rows re-read : {check.count()}")
(check.filter(F.col("cluster_id").isNotNull())
      .groupBy("cluster_id", "cluster_label").count()
      .orderBy("cluster_id").show(truncate=False))

# Do the clusters actually separate on the metric that matters?
(check.filter(F.col("cluster_id").isNotNull())
      .groupBy("cluster_label")
      .agg(F.round(F.min("delay_rate"), 1).alias("min_delay_rate"),
           F.round(F.avg("delay_rate"), 1).alias("avg_delay_rate"),
           F.round(F.max("delay_rate"), 1).alias("max_delay_rate"))
      .orderBy("avg_delay_rate").show(truncate=False))

### Limitations to record in the report

- **Silhouette is modest.** Airport operational characteristics form a continuum, not
  naturally separated groups. K-Means imposes boundaries on that continuum; the clusters
  are a useful summary, not a discovery of latent categories.
- **k is a judgement call.** Silhouette alone favours small k; interpretability favours
  enough clusters to distinguish airport types. Both criteria are reported so the choice
  is auditable rather than asserted.
- **Volume dominates without scaling.** Standardisation is what stops this from being a
  size ranking. That is a modelling decision, and a different scaling choice would produce
  different clusters.
- **Small airports are excluded by design.** Clustering a 200-flight airport would produce
  a confident but noise-driven label. They keep all metrics and are shown uncoloured on
  the dashboard map, with volume context — per the fairness commitments in proposal §18.

In [ ]:
for df in (clustering_input, scaled, clustered):
    df.unpersist()
spark.stop()
print("Notebook 07 complete.")